# Claude Code Co-Programming Effectiveness Analysis

**Robotics Project** is a ~117K LOC production robotics project (ROS2, CUDA vision, ML prediction, real-time control) built by a single developer co-programming with Claude throughout the entire project — 10 months, 554 commits (Mar 2025 - Feb 2026).

This notebook quantifies how effective the "1 dev + Claude" co-programming approach was compared to traditional team-based development, using both **top-down** (codebase review) and **bottom-up** (weekly commit aggregates) analysis.

---
## Section 1: Setup & Git Data Extraction

In [1]:
import subprocess
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from datetime import datetime, timedelta
import re
import os
import warnings
warnings.filterwarnings('ignore')

pio.templates.default = 'plotly_white'

# Colors
COLORS = {
    'primary': '#2563EB',
    'secondary': '#7C3AED',
    'accent': '#059669',
    'warning': '#D97706',
    'danger': '#DC2626',
    'light': '#F3F4F6',
    'claude_web': '#D97706',
    'claude_code': '#7C3AED',
}

DOMAIN_COLORS = {
    'vision': '#2563EB',
    'control': '#DC2626',
    'game_logic': '#F59E0B',
    'prediction': '#059669',
    'ml': '#7C3AED',
    'depth_vision': '#0891B2',
    'msgs': '#6B7280',
    'bringup': '#EC4899',
    'visualization': '#8B5CF6',
    'config': '#78716C',
    'other': '#9CA3AF',
}

REPO_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    REPO_DIR = os.getcwd()
    while REPO_DIR != '/' and not os.path.exists(os.path.join(REPO_DIR, '.git')):
        REPO_DIR = os.path.dirname(REPO_DIR)

print(f'Repository: {REPO_DIR}')

Repository: /Users/denn/robotics_ws


In [2]:
# Extract full git log with numstat (main branch only)
DELIM = '<<<COMMIT>>>'
fmt = f'{DELIM}%n%H%n%ai%n%an%n%s%n%b%n<<<BODY_END>>>'

result = subprocess.run(
    ['git', 'log', f'--format={fmt}', '--numstat'],
    capture_output=True, text=True, cwd=REPO_DIR
)

raw_commits = result.stdout.split(DELIM)[1:]  # skip first empty

commits = []
for raw in raw_commits:
    lines = raw.strip().split('\n')
    if len(lines) < 5:
        continue
    
    hash_val = lines[0].strip()
    date_str = lines[1].strip()
    author = lines[2].strip()
    subject = lines[3].strip()
    
    # Extract body (between subject and <<<BODY_END>>>)
    body_end_idx = None
    for i, l in enumerate(lines):
        if '<<<BODY_END>>>' in l:
            body_end_idx = i
            break
    
    if body_end_idx is not None:
        body = '\n'.join(lines[4:body_end_idx]).strip()
        numstat_lines = lines[body_end_idx+1:]
    else:
        body = '\n'.join(lines[4:]).strip()
        numstat_lines = []
    
    # Parse numstat
    insertions = 0
    deletions = 0
    files_changed = []
    for ns in numstat_lines:
        ns = ns.strip()
        if not ns:
            continue
        parts = ns.split('\t')
        if len(parts) >= 3:
            ins = int(parts[0]) if parts[0] != '-' else 0
            dels = int(parts[1]) if parts[1] != '-' else 0
            insertions += ins
            deletions += dels
            files_changed.append(parts[2])
    
    # Detect co-author tag
    has_coauthor = 'Co-Authored-By' in body
    
    # Parse date
    dt = pd.to_datetime(date_str)
    
    commits.append({
        'hash': hash_val,
        'date': dt,
        'author': author,
        'subject': subject,
        'body': body,
        'has_coauthor_tag': has_coauthor,
        'insertions': insertions,
        'deletions': deletions,
        'net_loc': insertions - deletions,
        'files_changed': files_changed,
        'n_files': len(files_changed),
    })

df = pd.DataFrame(commits)
df = df.sort_values('date').reset_index(drop=True)

# Tag claude tool usage
df['claude_tool'] = df['has_coauthor_tag'].map({True: 'Claude Code CLI', False: 'Claude Web UI'})

print(f'Total commits: {len(df)}')
print(f'Date range: {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'Co-authored (Claude Code CLI): {df["has_coauthor_tag"].sum()}')
print(f'Web UI assisted: {(~df["has_coauthor_tag"]).sum()}')
print(f'Total insertions: {df["insertions"].sum():,}')
print(f'Total deletions: {df["deletions"].sum():,}')
print(f'Net LOC: {df["net_loc"].sum():,}')

Total commits: 554
Date range: 2025-03-30 to 2026-02-04
Co-authored (Claude Code CLI): 86
Web UI assisted: 468
Total insertions: 430,583
Total deletions: 138,816
Net LOC: 291,767


---
## Section 2: Commit Classification

In [3]:
# Domain classification based on file paths
DOMAIN_PATTERNS = {
    'vision': r'vision/',
    'control': r'control/.*\.(cpp|hpp|h)',
    'game_logic': r'control/.*\.py',
    'prediction': r'prediction/',
    'ml': r'ml/',
    'depth_vision': r'depth_vision/',
    'msgs': r'msgs/',
    'bringup': r'bringup/',
    'visualization': r'visualization/',
    'config': r'description/',
}

for domain, pattern in DOMAIN_PATTERNS.items():
    df[f'domain_{domain}'] = df['files_changed'].apply(
        lambda files: any(re.search(pattern, f) for f in files)
    )

# 'other' domain: no specific domain matched
domain_cols = [c for c in df.columns if c.startswith('domain_')]
df['domain_other'] = ~df[domain_cols].any(axis=1)

# Primary domain (first match)
def get_primary_domain(row):
    for domain in DOMAIN_PATTERNS.keys():
        if row[f'domain_{domain}']:
            return domain
    return 'other'

df['primary_domain'] = df.apply(get_primary_domain, axis=1)

# Count domains per commit
all_domain_cols = [c for c in df.columns if c.startswith('domain_')]
df['n_domains'] = df[all_domain_cols].sum(axis=1)

# Commit type classification by message heuristics
def classify_commit_type(subject):
    s = subject.lower()
    if any(w in s for w in ['fix', 'bug', 'correct', 'patch', 'resolve', 'issue']):
        return 'bugfix'
    elif any(w in s for w in ['refactor', 'restructur', 'reorganiz', 'clean', 'rename', 'extract', 'abstract', 'simplif']):
        return 'refactor'
    elif any(w in s for w in ['config', 'param', 'tuning', 'tune', 'adjust', 'threshold', 'calibrat', 'yaml', 'settings']):
        return 'config/tuning'
    elif any(w in s for w in ['doc', 'readme', 'comment', 'claude.md']):
        return 'docs'
    elif any(w in s for w in ['build', 'cmake', 'depend', 'package.xml', 'colcon']):
        return 'build'
    else:
        return 'feature'

df['commit_type'] = df['subject'].apply(classify_commit_type)

print('=== Domain Distribution ===')
print(df['primary_domain'].value_counts())
print(f'\nMulti-domain commits: {(df["n_domains"] > 1).sum()}')
print(f'\n=== Commit Type Distribution ===')
print(df['commit_type'].value_counts())

=== Domain Distribution ===
primary_domain
control          210
vision           126
game_logic        37
visualization     36
config            35
other             34
prediction        27
depth_vision      20
ml                15
bringup           13
msgs               1
Name: count, dtype: int64

Multi-domain commits: 201

=== Commit Type Distribution ===
commit_type
feature          384
bugfix           102
config/tuning     39
refactor          21
docs               6
build              2
Name: count, dtype: int64


---
## Section 3: Weekly Aggregation

In [4]:
# Group by ISO week
# Normalize to UTC then strip timezone for consistent weekly grouping
df['date'] = pd.to_datetime(df['date'], utc=True).dt.tz_localize(None)
isocal = df['date'].dt.isocalendar()
df['iso_year'] = isocal['year'].astype(int)
df['iso_week'] = isocal['week'].astype(int)
df['year_week'] = df['iso_year'].astype(str) + '-W' + df['iso_week'].astype(str).str.zfill(2)
df['week_start'] = df['date'].dt.to_period('W').apply(lambda r: r.start_time)

# Weekly aggregation
weekly = df.groupby('year_week').agg(
    week_start=('week_start', 'first'),
    commits=('hash', 'count'),
    insertions=('insertions', 'sum'),
    deletions=('deletions', 'sum'),
    net_loc=('net_loc', 'sum'),
    files_changed=('n_files', 'sum'),
    coauthor_commits=('has_coauthor_tag', 'sum'),
).reset_index()

weekly['web_ui_commits'] = weekly['commits'] - weekly['coauthor_commits']
weekly = weekly.sort_values('week_start').reset_index(drop=True)

# Domain breakdown per week
for domain in list(DOMAIN_PATTERNS.keys()) + ['other']:
    col = f'domain_{domain}'
    domain_weekly = df[df[col]].groupby('year_week').agg(
        **{f'{domain}_commits': ('hash', 'count'),
           f'{domain}_insertions': ('insertions', 'sum')}
    ).reset_index()
    weekly = weekly.merge(domain_weekly, on='year_week', how='left')
    weekly[f'{domain}_commits'] = weekly[f'{domain}_commits'].fillna(0).astype(int)
    weekly[f'{domain}_insertions'] = weekly[f'{domain}_insertions'].fillna(0).astype(int)

print(f'Total weeks with activity: {len(weekly)}')
print(f'Average commits/week: {weekly["commits"].mean():.1f}')
print(f'Average net LOC/week: {weekly["net_loc"].mean():.0f}')
weekly.head()

Total weeks with activity: 44
Average commits/week: 12.6
Average net LOC/week: 6631


,year_week,week_start,commits,insertions,deletions,net_loc,files_changed,coauthor_commits,web_ui_commits,vision_commits,...,msgs_commits,msgs_insertions,bringup_commits,bringup_insertions,visualization_commits,visualization_insertions,config_commits,config_insertions,other_commits,other_insertions
0,2025-W13,2025-03-24,2,814,0,814,15,0,2,0,...,0,0,0,0,0,0,1,562,1,252
1,2025-W15,2025-04-07,5,4234,1694,2540,116,0,5,0,...,0,0,0,0,0,0,4,3836,1,398
2,2025-W16,2025-04-14,10,5543,1744,3799,89,0,10,0,...,0,0,0,0,0,0,10,5543,0,0
3,2025-W17,2025-04-21,14,3653,3020,633,161,0,14,0,...,0,0,0,0,0,0,13,2978,0,0
4,2025-W18,2025-04-28,6,2449,827,1622,43,0,6,3,...,1,809,0,0,0,0,4,1115,0,0


---
## Section 4: Top-Down Analysis — Codebase Composition

In [5]:
# LOC by package (from actual file counts)
import glob as glob_mod

packages = [
    'vision', 'control', 'prediction',
    'depth_vision', 'msgs', 'bringup',
    'visualization', 'description', 'ml'
]

pkg_data = []
for pkg in packages:
    pkg_dir = os.path.join(REPO_DIR, 'src', 'robotics_project', pkg)
    if not os.path.exists(pkg_dir):
        continue
    
    cpp_loc = 0
    py_loc = 0
    other_loc = 0
    
    for root, dirs, files in os.walk(pkg_dir):
        for f in files:
            fpath = os.path.join(root, f)
            try:
                with open(fpath, 'r', errors='ignore') as fh:
                    loc = sum(1 for _ in fh)
            except:
                continue
            
            if f.endswith(('.cpp', '.hpp', '.h')):
                cpp_loc += loc
            elif f.endswith('.py'):
                py_loc += loc
            elif f.endswith(('.yaml', '.yml', '.xml', '.urdf', '.xacro', '.msg', '.srv', '.action', '.launch.py')):
                # .launch.py counted as Python
                if f.endswith('.launch.py'):
                    py_loc += loc
                else:
                    other_loc += loc
    
    pkg_data.append({
        'package': pkg.replace('robotics_project_', ''),
        'C++': cpp_loc,
        'Python': py_loc,
        'Config/Other': other_loc,
        'total': cpp_loc + py_loc + other_loc,
    })

pkg_df = pd.DataFrame(pkg_data).sort_values('total', ascending=True)

# Horizontal bar chart: LOC by package split by language
fig = go.Figure()
fig.add_trace(go.Bar(
    y=pkg_df['package'], x=pkg_df['C++'],
    name='C++', orientation='h',
    marker_color='#2563EB'
))
fig.add_trace(go.Bar(
    y=pkg_df['package'], x=pkg_df['Python'],
    name='Python', orientation='h',
    marker_color='#059669'
))
fig.add_trace(go.Bar(
    y=pkg_df['package'], x=pkg_df['Config/Other'],
    name='Config/Other', orientation='h',
    marker_color='#9CA3AF'
))
fig.update_layout(
    title='Lines of Code by Package',
    xaxis_title='Lines of Code',
    barmode='stack',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [6]:
# Treemap: package -> language breakdown
treemap_data = []
for _, row in pkg_df.iterrows():
    for lang in ['C++', 'Python', 'Config/Other']:
        if row[lang] > 0:
            treemap_data.append({
                'package': row['package'],
                'language': lang,
                'loc': row[lang],
                'label': f"{row['package']}/{lang}"
            })

tree_df = pd.DataFrame(treemap_data)
fig = px.treemap(
    tree_df, path=['package', 'language'], values='loc',
    title='Codebase Composition Treemap',
    color='package',
    color_discrete_map={
        'vision': '#2563EB', 'control': '#DC2626', 'prediction': '#059669',
        'ml': '#7C3AED', 'depth_vision': '#0891B2', 'visualization': '#8B5CF6',
        'bringup': '#EC4899', 'description': '#78716C', 'msgs': '#6B7280'
    }
)
fig.update_layout(height=500)
fig.show()

In [7]:
# Key complexity indicators table
total_cpp = pkg_df['C++'].sum()
total_py = pkg_df['Python'].sum()
total_other = pkg_df['Config/Other'].sum()
total_loc = pkg_df['total'].sum()

complexity_data = {
    'Metric': [
        'Total Lines of Code', 'C++ LOC', 'Python LOC', 'Config/Other LOC',
        'Number of Packages', 'Total Commits', 'Project Duration (months)',
        'Developers', 'Co-Author Tagged Commits (Claude Code)',
    ],
    'Value': [
        f'{total_loc:,}', f'{total_cpp:,}', f'{total_py:,}', f'{total_other:,}',
        str(len(packages)), str(len(df)), '10',
        '1', str(df['has_coauthor_tag'].sum()),
    ]
}

complexity_df = pd.DataFrame(complexity_data)

fig = go.Figure(data=[go.Table(
    header=dict(values=['Metric', 'Value'],
                fill_color='#2563EB', font=dict(color='white', size=13),
                align='left'),
    cells=dict(values=[complexity_df['Metric'], complexity_df['Value']],
               fill_color='#F3F4F6', align='left', font=dict(size=12),
               height=28))
])
fig.update_layout(title='Key Codebase Metrics', height=380)
fig.show()

---
## Section 5: Top-Down Analysis — Traditional Team Estimate

In [8]:
# Traditional team roles and estimates
# Based on industry benchmarks for complex robotics:
# - C++/CUDA: ~20 productive LOC/day
# - Python/ROS: ~40 LOC/day
# - ~22 working days/month

roles = [
    {
        'role': 'Senior Robotics Engineer',
        'scope': 'C++ control, MoveIt2, motion planning',
        'loc': 18219 + 1228,  # control C++ + config
        'pm_low': 8, 'pm_high': 10,
        'color': '#DC2626'
    },
    {
        'role': 'Computer Vision Engineer',
        'scope': 'CUDA pipeline, Kalman filters, tracking',
        'loc': 11706 + 591 + 856,
        'pm_low': 5, 'pm_high': 7,
        'color': '#2563EB'
    },
    {
        'role': 'ML Engineer',
        'scope': 'Training pipeline, ONNX, prediction',
        'loc': 35611 + 467 + 10854 + 527 + 520,
        'pm_low': 6, 'pm_high': 8,
        'color': '#7C3AED'
    },
    {
        'role': 'Depth Vision Engineer',
        'scope': 'depth sensor integration, 3D detection',
        'loc': 5035 + 2569 + 257,
        'pm_low': 3, 'pm_high': 4,
        'color': '#0891B2'
    },
    {
        'role': 'ROS2 Systems Integrator',
        'scope': 'Launch files, msgs, bringup, deployment',
        'loc': 1645 + 858 + 503 + 1123 + 1986,
        'pm_low': 3, 'pm_high': 4,
        'color': '#EC4899'
    },
    {
        'role': 'UI/Visualization Dev',
        'scope': 'Control center GUI, RViz, game display, game logic',
        'loc': 7846 + 3009 + 550 + 11883,
        'pm_low': 3, 'pm_high': 4,
        'color': '#8B5CF6'
    },
]

roles_df = pd.DataFrame(roles)
roles_df['pm_mid'] = (roles_df['pm_low'] + roles_df['pm_high']) / 2

total_pm_low = roles_df['pm_low'].sum()
total_pm_high = roles_df['pm_high'].sum()
total_pm_mid = roles_df['pm_mid'].sum()

print(f'Traditional team estimate: {total_pm_low}-{total_pm_high} person-months (mid: {total_pm_mid})')
print(f'Actual: 1 developer x 10 months = 10 person-months')
print(f'Multiplier: {total_pm_mid/10:.1f}x')

Traditional team estimate: 28-37 person-months (mid: 32.5)
Actual: 1 developer x 10 months = 10 person-months
Multiplier: 3.2x


In [9]:
# Stacked bar: traditional team composition
fig = go.Figure()
for _, row in roles_df.iterrows():
    fig.add_trace(go.Bar(
        x=[row['pm_mid']],
        y=[row['role']],
        orientation='h',
        name=row['role'],
        marker_color=row['color'],
        text=f"{row['pm_low']}-{row['pm_high']} PM",
        textposition='inside',
        hovertemplate=f"{row['role']}<br>Scope: {row['scope']}<br>LOC: {row['loc']:,}<br>Est: {row['pm_low']}-{row['pm_high']} person-months<extra></extra>"
    ))

fig.update_layout(
    title='Traditional Team — Estimated Person-Months by Role',
    xaxis_title='Person-Months',
    showlegend=False,
    height=400,
)
fig.show()

In [10]:
# Gantt-style: traditional parallel development timeline
# Assumes 4-5 engineers working in parallel, with dependencies
gantt_data = [
    {'role': 'ROS2 Systems Integrator', 'start': 0, 'end': 4, 'color': '#EC4899'},
    {'role': 'Senior Robotics Engineer', 'start': 1, 'end': 11, 'color': '#DC2626'},
    {'role': 'Computer Vision Engineer', 'start': 1, 'end': 8, 'color': '#2563EB'},
    {'role': 'ML Engineer', 'start': 2, 'end': 10, 'color': '#7C3AED'},
    {'role': 'Depth Vision Engineer', 'start': 4, 'end': 8, 'color': '#0891B2'},
    {'role': 'UI/Visualization Dev', 'start': 3, 'end': 7, 'color': '#8B5CF6'},
    {'role': 'Integration & Testing', 'start': 8, 'end': 13, 'color': '#6B7280'},
]

fig = go.Figure()
for item in gantt_data:
    fig.add_trace(go.Bar(
        y=[item['role']],
        x=[item['end'] - item['start']],
        base=[item['start']],
        orientation='h',
        marker_color=item['color'],
        name=item['role'],
        text=f"Mo {item['start']+1}-{item['end']}",
        textposition='inside',
        showlegend=False,
    ))

fig.update_layout(
    title='Traditional Team — Parallel Development Timeline (Est. 13 months, 4-5 engineers)',
    xaxis_title='Months',
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    height=380,
)
fig.show()

In [11]:
# Comparison bar: traditional vs actual person-months
fig = go.Figure()

categories = ['Traditional Team\n(estimated)', 'Actual\n(1 dev + Claude)']
values = [total_pm_mid, 10]
colors = ['#6B7280', '#7C3AED']

fig.add_trace(go.Bar(
    x=categories,
    y=values,
    marker_color=colors,
    text=[f'{total_pm_mid:.0f} PM\n({int(total_pm_low)}-{int(total_pm_high)} range)', '10 PM'],
    textposition='outside',
    textfont=dict(size=14),
))

fig.update_layout(
    title=f'Total Person-Months: Traditional Team vs 1 Dev + Claude ({total_pm_mid/10:.1f}x multiplier)',
    yaxis_title='Person-Months',
    height=400,
    showlegend=False,
)
fig.show()

---
## Section 6: Bottom-Up Analysis — Weekly Development Velocity

In [12]:
# Line chart: commits per week with trend line
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['commits'],
    mode='lines+markers',
    name='Commits/week',
    line=dict(color=COLORS['primary'], width=2),
    marker=dict(size=5),
))

# Trend line (rolling average)
weekly['commits_trend'] = weekly['commits'].rolling(4, center=True, min_periods=1).mean()
fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['commits_trend'],
    mode='lines',
    name='4-week trend',
    line=dict(color=COLORS['danger'], width=3, dash='dash'),
))

fig.update_layout(
    title='Commits per Week Over Time',
    xaxis_title='Date', yaxis_title='Commits',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [13]:
# Line chart: net LOC per week
fig = go.Figure()

fig.add_trace(go.Bar(
    x=weekly['week_start'], y=weekly['insertions'],
    name='Insertions', marker_color='#059669', opacity=0.6,
))
fig.add_trace(go.Bar(
    x=weekly['week_start'], y=-weekly['deletions'],
    name='Deletions', marker_color='#DC2626', opacity=0.6,
))
fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['net_loc'],
    mode='lines+markers',
    name='Net LOC',
    line=dict(color=COLORS['primary'], width=2),
    marker=dict(size=4),
))

fig.update_layout(
    title='Weekly Code Changes (Insertions, Deletions, Net LOC)',
    xaxis_title='Date', yaxis_title='Lines of Code',
    barmode='relative',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [14]:
# Stacked area: LOC by domain per week
domain_list = ['vision', 'control', 'game_logic', 'prediction', 'ml',
               'depth_vision', 'msgs', 'bringup', 'visualization', 'config', 'other']

fig = go.Figure()
for domain in domain_list:
    col = f'{domain}_insertions'
    if col in weekly.columns:
        fig.add_trace(go.Scatter(
            x=weekly['week_start'], y=weekly[col],
            mode='lines',
            name=domain,
            stackgroup='one',
            line=dict(width=0.5),
            fillcolor=DOMAIN_COLORS.get(domain, '#9CA3AF'),
            marker_color=DOMAIN_COLORS.get(domain, '#9CA3AF'),
        ))

fig.update_layout(
    title='Weekly Code Insertions by Domain (Development Focus Over Time)',
    xaxis_title='Date', yaxis_title='Lines Inserted',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
)
fig.show()

In [15]:
# Annotated milestones timeline
# Detect major milestones from commit messages
milestones = []
milestone_keywords = {
    'Initial commit': 'Project Start',
    'MoveIt': 'MoveIt2 Integration',
    'vision': 'Vision Pipeline',
    'play mode': 'Play Mode',
    'game_control': 'Game Control',
    'prediction': 'Prediction System',
    'depth': 'Depth Vision',
    'museum': 'Production Environment',
    'ONNX': 'ML Model Deploy',
    'movement_queue': 'Movement Queue',
    'kalman': 'Kalman Tracking',
}

seen_milestones = set()
for _, row in df.iterrows():
    for kw, label in milestone_keywords.items():
        if kw.lower() in row['subject'].lower() and label not in seen_milestones:
            milestones.append({'date': row['date'], 'label': label})
            seen_milestones.add(label)
            break

# Cumulative LOC over time
weekly['cumulative_loc'] = weekly['net_loc'].cumsum()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['cumulative_loc'],
    mode='lines',
    fill='tozeroy',
    name='Cumulative Net LOC',
    line=dict(color=COLORS['primary'], width=2),
    fillcolor='rgba(37, 99, 235, 0.1)',
))

# Add milestone annotations
for ms in milestones:
    ms_date = pd.Timestamp(ms['date'])
    diffs = (weekly['week_start'] - ms_date).abs()
    closest_idx = diffs.argsort().iloc[0]
    y_val = weekly.loc[closest_idx, 'cumulative_loc']
    fig.add_annotation(
        x=ms_date, y=y_val,
        text=ms['label'],
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowcolor='#6B7280',
        font=dict(size=9),
        ax=0, ay=-35,
    )

fig.update_layout(
    title='Cumulative Net LOC with Development Milestones',
    xaxis_title='Date', yaxis_title='Cumulative Net LOC',
    height=500,
)
fig.show()

---
## Section 7: Bottom-Up Analysis — Weekly Effort Estimation

In [16]:
# Effort estimation constants
# These estimate how many hours a traditional engineer would spend per commit,
# based on the size and type of changes. We use a log-based scaling since
# effort doesn't scale linearly with LOC — a 100-line change doesn't take
# 10x longer than a 10-line change.
EFFORT_CONSTANTS = {
    'cpp_base_hours': 2.0,           # Base hours for any C++ commit
    'cpp_hours_per_loc': 0.03,       # Additional hours per LOC changed (C++)
    'python_base_hours': 1.0,        # Base hours for any Python commit
    'python_hours_per_loc': 0.015,   # Additional hours per LOC changed (Python)
    'config_base_hours': 0.5,        # Base hours for config changes
    'config_hours_per_loc': 0.005,   # Additional hours per LOC changed (config)
    'integration_multiplier': 1.3,   # Cross-package integration overhead
    'test_debug_multiplier': 1.5,    # Testing and debugging overhead
    'max_hours_per_commit': 40,      # Cap: no single commit > 1 work-week
    'senior_engineer_daily_rate': 800,  # USD/day for cost comparison
    'claude_code_monthly_cost': 200,    # USD/month subscription
}

def estimate_traditional_hours(row):
    """Estimate how many hours a traditional team would spend on this commit."""
    if row['n_files'] == 0:
        return 0
    
    total_loc = row['insertions'] + row['deletions']
    
    # Estimate language split from files
    cpp_files = sum(1 for f in row['files_changed'] if f.endswith(('.cpp', '.hpp', '.h')))
    py_files = sum(1 for f in row['files_changed'] if f.endswith('.py'))
    config_files = sum(1 for f in row['files_changed'] 
                       if f.endswith(('.yaml', '.yml', '.xml', '.urdf', '.xacro', '.msg', '.srv')))
    other_files = max(row['n_files'] - cpp_files - py_files - config_files, 0)
    
    total_cat = cpp_files + py_files + config_files + other_files
    if total_cat == 0:
        total_cat = 1
    
    cpp_frac = cpp_files / total_cat
    py_frac = (py_files + other_files) / total_cat  # other defaults to python rate
    config_frac = config_files / total_cat
    
    # Use log scaling for LOC: diminishing effort per additional line
    loc_capped = min(total_loc, 2000)  # soft cap on extreme outliers
    
    hours = 0
    if cpp_frac > 0:
        hours += cpp_frac * (EFFORT_CONSTANTS['cpp_base_hours'] + 
                             loc_capped * EFFORT_CONSTANTS['cpp_hours_per_loc'])
    if py_frac > 0:
        hours += py_frac * (EFFORT_CONSTANTS['python_base_hours'] + 
                            loc_capped * EFFORT_CONSTANTS['python_hours_per_loc'])
    if config_frac > 0:
        hours += config_frac * (EFFORT_CONSTANTS['config_base_hours'] + 
                                loc_capped * EFFORT_CONSTANTS['config_hours_per_loc'])
    
    # Integration overhead for multi-domain commits
    if row['n_domains'] > 1:
        hours *= EFFORT_CONSTANTS['integration_multiplier']
    
    # Testing/debugging overhead
    hours *= EFFORT_CONSTANTS['test_debug_multiplier']
    
    # Hard cap
    hours = min(hours, EFFORT_CONSTANTS['max_hours_per_commit'])
    
    return hours

df['trad_hours_est'] = df.apply(estimate_traditional_hours, axis=1)

# Weekly aggregation of effort estimates
effort_weekly = df.groupby('year_week').agg(
    trad_hours=('trad_hours_est', 'sum'),
    week_start=('week_start', 'first'),
).reset_index().sort_values('week_start')

weekly = weekly.merge(effort_weekly[['year_week', 'trad_hours']], on='year_week', how='left')
weekly['trad_hours'] = weekly['trad_hours'].fillna(0)
weekly['actual_hours'] = 40  # assumed 40h/week

total_trad = weekly['trad_hours'].sum()
total_actual = weekly['actual_hours'].sum()

print(f'Total estimated traditional hours: {total_trad:,.0f}')
print(f'Total actual hours (40h/week x {len(weekly)} weeks): {total_actual:,}')
print(f'Overall bottom-up multiplier: {total_trad / total_actual:.1f}x')
print(f'\\nPer-commit stats:')
print(f'  Median: {df["trad_hours_est"].median():.1f} hours')
print(f'  Mean: {df["trad_hours_est"].mean():.1f} hours')
print(f'  Max: {df["trad_hours_est"].max():.1f} hours (hard cap: {EFFORT_CONSTANTS["max_hours_per_commit"]}h)')
print(f'  Total commits: {len(df)}')

Total estimated traditional hours: 8,526
Total actual hours (40h/week x 44 weeks): 1,760
Overall bottom-up multiplier: 4.8x
\nPer-commit stats:
  Median: 10.3 hours
  Mean: 15.4 hours
  Max: 40.0 hours (hard cap: 40h)
  Total commits: 554


In [17]:
# Dual-axis line chart: estimated traditional hours/week vs actual 40h/week
fig = make_subplots(specs=[[{'secondary_y': False}]])

fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['trad_hours'],
    mode='lines',
    name='Est. Traditional Hours/Week',
    line=dict(color=COLORS['danger'], width=2),
    fill='tozeroy',
    fillcolor='rgba(220, 38, 38, 0.1)',
))

fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['actual_hours'],
    mode='lines',
    name='Actual Hours/Week (40h)',
    line=dict(color=COLORS['accent'], width=2, dash='dash'),
))

fig.update_layout(
    title='Weekly Effort: Traditional Estimate vs Actual (1 dev + Claude)',
    xaxis_title='Date', yaxis_title='Hours/Week',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [18]:
# Cumulative hours comparison
weekly['cum_trad_hours'] = weekly['trad_hours'].cumsum()
weekly['cum_actual_hours'] = weekly['actual_hours'].cumsum()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['cum_trad_hours'],
    mode='lines',
    name='Cumulative Traditional Hours (est.)',
    fill='tozeroy',
    line=dict(color=COLORS['danger'], width=2),
    fillcolor='rgba(220, 38, 38, 0.1)',
))

fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['cum_actual_hours'],
    mode='lines',
    name='Cumulative Actual Hours (40h/week)',
    fill='tozeroy',
    line=dict(color=COLORS['accent'], width=2),
    fillcolor='rgba(5, 150, 105, 0.1)',
))

fig.update_layout(
    title='Cumulative Hours: Traditional Estimate vs Actual',
    xaxis_title='Date', yaxis_title='Cumulative Hours',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [19]:
# Rolling productivity multiplier
weekly['productivity_mult'] = weekly['trad_hours'] / weekly['actual_hours']
weekly['productivity_mult_smooth'] = weekly['productivity_mult'].rolling(4, center=True, min_periods=1).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['productivity_mult'],
    mode='markers',
    name='Weekly Multiplier',
    marker=dict(color=COLORS['primary'], size=5, opacity=0.4),
))

fig.add_trace(go.Scatter(
    x=weekly['week_start'], y=weekly['productivity_mult_smooth'],
    mode='lines',
    name='4-week Smoothed Multiplier',
    line=dict(color=COLORS['secondary'], width=3),
))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='1x (break even)')

fig.update_layout(
    title='Rolling Productivity Multiplier (Traditional Estimate / Actual)',
    xaxis_title='Date', yaxis_title='Productivity Multiplier',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

---
## Section 8: Claude Tooling Evolution

In [20]:
# Monthly breakdown of Claude tooling
df['month'] = df['date'].dt.to_period('M')
monthly_tool = df.groupby(['month', 'claude_tool']).size().unstack(fill_value=0).reset_index()
monthly_tool['month_str'] = monthly_tool['month'].astype(str)

fig = go.Figure()

if 'Claude Web UI' in monthly_tool.columns:
    fig.add_trace(go.Bar(
        x=monthly_tool['month_str'], y=monthly_tool['Claude Web UI'],
        name='Claude Web UI',
        marker_color=COLORS['claude_web'],
    ))

if 'Claude Code CLI' in monthly_tool.columns:
    fig.add_trace(go.Bar(
        x=monthly_tool['month_str'], y=monthly_tool['Claude Code CLI'],
        name='Claude Code CLI',
        marker_color=COLORS['claude_code'],
    ))

fig.update_layout(
    title='Claude Tooling Evolution — Commits per Month',
    xaxis_title='Month', yaxis_title='Commits',
    barmode='stack',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [21]:
# Determine transition point
first_claude_code = df[df['has_coauthor_tag']]['date'].min()
print(f'First Claude Code CLI commit: {first_claude_code}')

df['phase'] = df['date'].apply(lambda d: 'Claude Code CLI' if d >= first_claude_code else 'Claude Web UI')

# Compare phases
phase_stats = df.groupby('phase').agg(
    commits=('hash', 'count'),
    total_insertions=('insertions', 'sum'),
    total_deletions=('deletions', 'sum'),
    avg_insertions=('insertions', 'mean'),
    avg_deletions=('deletions', 'mean'),
    avg_files=('n_files', 'mean'),
    avg_net_loc=('net_loc', 'mean'),
).reset_index()

print('\n=== Phase Comparison ===')
print(phase_stats.to_string(index=False))

First Claude Code CLI commit: 2025-10-20 10:40:52

=== Phase Comparison ===
          phase  commits  total_insertions  total_deletions  avg_insertions  avg_deletions  avg_files  avg_net_loc
Claude Code CLI      260            130303            27966      501.165385     107.561538   5.473077   393.603846
  Claude Web UI      294            300280           110850     1021.360544     377.040816   7.017007   644.319728


In [22]:
# Velocity comparison before/after Claude Code
weekly['phase'] = weekly['week_start'].apply(
    lambda d: 'Claude Code CLI' if d >= first_claude_code else 'Claude Web UI'
)

fig = go.Figure()

for phase, color in [('Claude Web UI', COLORS['claude_web']), ('Claude Code CLI', COLORS['claude_code'])]:
    mask = weekly['phase'] == phase
    fig.add_trace(go.Scatter(
        x=weekly.loc[mask, 'week_start'],
        y=weekly.loc[mask, 'commits'],
        mode='lines+markers',
        name=phase,
        line=dict(color=color, width=2),
        marker=dict(size=5),
    ))

# Use string for vline to avoid plotly Timestamp arithmetic issue
fig.add_shape(
    type='line',
    x0=str(first_claude_code), x1=str(first_claude_code),
    y0=0, y1=1, yref='paper',
    line=dict(dash='dash', color='gray'),
)
fig.add_annotation(
    x=str(first_claude_code), y=1, yref='paper',
    text='Claude Code CLI adoption',
    showarrow=False, font=dict(size=10, color='gray'),
    yshift=10,
)

fig.update_layout(
    title='Commit Velocity: Before vs After Claude Code CLI Adoption',
    xaxis_title='Date', yaxis_title='Commits/Week',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [23]:
# Average commit size comparison
phase_avg = df.groupby('phase').agg(
    avg_loc_changed=('insertions', lambda x: (x + df.loc[x.index, 'deletions']).mean()),
    avg_insertions=('insertions', 'mean'),
    avg_files=('n_files', 'mean'),
).reset_index()

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'Avg Lines Changed/Commit', 'Avg Insertions/Commit', 'Avg Files/Commit'
])

colors_phase = [COLORS['claude_web'], COLORS['claude_code']]

fig.add_trace(go.Bar(
    x=phase_avg['phase'], y=phase_avg['avg_loc_changed'],
    marker_color=colors_phase, showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=phase_avg['phase'], y=phase_avg['avg_insertions'],
    marker_color=colors_phase, showlegend=False,
), row=1, col=2)

fig.add_trace(go.Bar(
    x=phase_avg['phase'], y=phase_avg['avg_files'],
    marker_color=colors_phase, showlegend=False,
), row=1, col=3)

fig.update_layout(
    title='Commit Size: Claude Web UI vs Claude Code CLI Phase',
    height=400,
)
fig.show()

---
## Section 9: Comparison Dashboard

In [24]:
# Side-by-side: Traditional team vs Actual
trad_months = 13  # from Gantt estimate
trad_engineers = 5

comparison_data = {
    'Metric': ['Team Size', 'Duration (months)', 'Person-Months', 'Total LOC Delivered'],
    'Traditional Team (est.)': [f'{trad_engineers} engineers', str(trad_months), f'{total_pm_mid:.0f}', f'{total_loc:,}'],
    '1 Dev + Claude': ['1 developer', '10', '10', f'{total_loc:,}'],
}

comp_df = pd.DataFrame(comparison_data)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Metric', 'Traditional Team (est.)', '1 Dev + Claude'],
        fill_color='#2563EB', font=dict(color='white', size=13),
        align='center'
    ),
    cells=dict(
        values=[comp_df['Metric'], comp_df['Traditional Team (est.)'], comp_df['1 Dev + Claude']],
        fill_color=[['#F3F4F6']*4, ['#FEE2E2']*4, ['#EDE9FE']*4],
        align='center', font=dict(size=13),
        height=32
    ))
])
fig.update_layout(title='Head-to-Head Comparison', height=280)
fig.show()

In [25]:
# Cost comparison
daily_rate = EFFORT_CONSTANTS['senior_engineer_daily_rate']
claude_monthly = EFFORT_CONSTANTS['claude_code_monthly_cost']

trad_cost = total_pm_mid * 22 * daily_rate  # PM * working days * daily rate
actual_cost = 10 * 22 * daily_rate + 10 * claude_monthly  # 1 dev salary + Claude subscription

fig = go.Figure()

fig.add_trace(go.Bar(
    x=['Traditional Team', '1 Dev + Claude'],
    y=[trad_cost, actual_cost],
    marker_color=['#6B7280', '#7C3AED'],
    text=[f'${trad_cost:,.0f}', f'${actual_cost:,.0f}'],
    textposition='outside',
    textfont=dict(size=14),
))

fig.update_layout(
    title=f'Estimated Total Cost Comparison ({trad_cost/actual_cost:.1f}x savings)',
    yaxis_title='Total Cost (USD)',
    height=400,
    showlegend=False,
    yaxis=dict(tickformat='$,.0f'),
)
fig.show()

print(f'Traditional team cost: ${trad_cost:,.0f}')
print(f'  ({total_pm_mid:.0f} PM x 22 days x ${daily_rate}/day)')
print(f'Actual cost: ${actual_cost:,.0f}')
print(f'  (10 PM x 22 days x ${daily_rate}/day + 10 mo x ${claude_monthly}/mo Claude)')
print(f'Cost multiplier: {trad_cost/actual_cost:.1f}x')

Traditional team cost: $572,000
  (32 PM x 22 days x $800/day)
Actual cost: $178,000
  (10 PM x 22 days x $800/day + 10 mo x $200/mo Claude)
Cost multiplier: 3.2x


In [26]:
# Radar chart: expertise domain coverage
domains_radar = ['C++ Robotics', 'Computer Vision\n(CUDA)', 'ML/DL', 'Depth Sensing',
                 'ROS2 Systems', 'UI/Visualization', 'Game Logic']

# Traditional: each specialist covers their domain well, less in others
traditional_coverage = [0.95, 0.95, 0.95, 0.90, 0.90, 0.85, 0.80]
# 1 dev + Claude: good coverage across all domains (Claude helps fill gaps)
actual_coverage = [0.85, 0.80, 0.85, 0.80, 0.85, 0.80, 0.90]

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=traditional_coverage + [traditional_coverage[0]],
    theta=domains_radar + [domains_radar[0]],
    fill='toself',
    name='Traditional Team (specialists)',
    line_color='#6B7280',
    fillcolor='rgba(107, 114, 128, 0.15)',
))

fig.add_trace(go.Scatterpolar(
    r=actual_coverage + [actual_coverage[0]],
    theta=domains_radar + [domains_radar[0]],
    fill='toself',
    name='1 Dev + Claude',
    line_color='#7C3AED',
    fillcolor='rgba(124, 58, 237, 0.15)',
))

fig.update_layout(
    title='Domain Expertise Coverage',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
)
fig.show()

In [27]:
# Productivity multiplier summary bar
pm_multiplier = total_pm_mid / 10
cost_multiplier = trad_cost / actual_cost
hours_multiplier = weekly['trad_hours'].sum() / weekly['actual_hours'].sum()
time_multiplier = trad_months / 10

multipliers = {
    'Person-Months': pm_multiplier,
    'Cost': cost_multiplier,
    'Bottom-Up Hours': hours_multiplier,
    'Timeline': time_multiplier,
}

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(multipliers.keys()),
    y=list(multipliers.values()),
    marker_color=['#2563EB', '#059669', '#DC2626', '#D97706'],
    text=[f'{v:.1f}x' for v in multipliers.values()],
    textposition='outside',
    textfont=dict(size=16, color='black'),
))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='1x (no advantage)')

fig.update_layout(
    title='Productivity Multiplier Summary (Higher = More Effective with Claude)',
    yaxis_title='Multiplier',
    height=420,
    showlegend=False,
)
fig.show()

---
## Section 10: Summary & Key Findings

In [28]:
# Summary table
overall_mult = np.mean([pm_multiplier, hours_multiplier, time_multiplier])

summary_data = {
    'Metric': [
        'Project Duration', 'Total Commits', 'Total LOC (in robotics_project/)',
        'C++ LOC', 'Python LOC', 'Config/Other LOC',
        'Packages', 'Active Weeks',
        'Avg Commits/Week', 'Avg Net LOC/Week',
        '',
        'Claude Web UI Commits', 'Claude Code CLI Commits',
        'First Claude Code Commit',
        '',
        'Traditional Team Est. (person-months)', 'Actual (person-months)',
        'Person-Month Multiplier',
        'Traditional Est. Cost', 'Actual Cost',
        'Cost Multiplier',
        'Bottom-Up Hours Multiplier',
        'Overall Avg Multiplier',
    ],
    'Value': [
        f"{df['date'].min().strftime('%b %Y')} - {df['date'].max().strftime('%b %Y')}",
        str(len(df)),
        f'{total_loc:,}',
        f'{total_cpp:,}',
        f'{total_py:,}',
        f'{total_other:,}',
        str(len(packages)),
        str(len(weekly)),
        f'{weekly["commits"].mean():.1f}',
        f'{weekly["net_loc"].mean():,.0f}',
        '',
        str(int((~df['has_coauthor_tag']).sum())),
        str(int(df['has_coauthor_tag'].sum())),
        first_claude_code.strftime('%Y-%m-%d'),
        '',
        f'{total_pm_low}-{total_pm_high} ({total_pm_mid:.0f} mid)',
        '10',
        f'{pm_multiplier:.1f}x',
        f'${trad_cost:,.0f}',
        f'${actual_cost:,.0f}',
        f'{cost_multiplier:.1f}x',
        f'{hours_multiplier:.1f}x',
        f'{overall_mult:.1f}x',
    ]
}

summary_df = pd.DataFrame(summary_data)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Metric', 'Value'],
        fill_color='#2563EB', font=dict(color='white', size=13),
        align='left'
    ),
    cells=dict(
        values=[summary_df['Metric'], summary_df['Value']],
        fill_color=[['#F3F4F6' if i % 2 == 0 else 'white' for i in range(len(summary_df))]]*2,
        align='left', font=dict(size=12),
        height=26
    ))
])
fig.update_layout(title='Summary of Key Metrics', height=700)
fig.show()

### Key Findings

1. **One developer + Claude built a production-grade ~117K LOC robotics system in 10 months** — a project that would traditionally require a team of 4-5 specialized engineers over 12-16 months.

2. **Broad domain coverage without specialists:** The codebase spans CUDA-accelerated computer vision, C++ real-time robot control with MoveIt2, Python ML/prediction systems, depth sensing, game logic, and visualization — domains that traditionally require separate specialists.

3. **Claude Code CLI adoption (Oct 2025)** marked a transition in workflow, enabling tighter integration of AI assistance directly in the development loop.

4. **The bottom-up weekly effort analysis** corroborates the top-down team estimate, with both approaches suggesting a significant productivity multiplier.

5. **Cost efficiency:** The single-developer + Claude approach is dramatically more cost-effective than a traditional team, with savings on both salary overhead and coordination costs.

### Caveats & Limitations

- **LOC as a productivity metric** has well-known limitations — it doesn't capture design quality, maintainability, or debugging time accurately.
- **Effort estimation constants** are based on industry averages for complex robotics software and may vary significantly by team experience and domain.
- **The "40 hours/week" assumption** for actual effort may not reflect reality — solo developers often work variable hours.
- **Traditional team estimates** don't account for coordination overhead (meetings, code reviews, onboarding), which would increase the traditional estimate further.
- **Not all commits are equal** — some represent major architectural changes while others are minor config tweaks. The analysis treats them uniformly by LOC.
- **Code quality comparison** is not included — a traditional team might produce more thoroughly tested and reviewed code.
- **External package code** (third-party ROS drivers for the manipulator and sensors) is excluded from the analysis as it was not developed in this project.

In [29]:
# Sensitivity analysis — adjust effort constants and see impact
print('=== Sensitivity Analysis ===')
print('\nAdjust EFFORT_CONSTANTS at the top of Section 7 and re-run to see impact.')
print('\nCurrent constants:')
for k, v in EFFORT_CONSTANTS.items():
    print(f'  {k}: {v}')

print(f'\n--- Current Results ---')
print(f'Top-down person-month multiplier: {pm_multiplier:.1f}x')
print(f'Bottom-up hours multiplier: {hours_multiplier:.1f}x')
print(f'Cost multiplier: {cost_multiplier:.1f}x')
print(f'Timeline multiplier: {time_multiplier:.1f}x')
print(f'Overall average: {overall_mult:.1f}x')

# Quick sensitivity on per-LOC rates
print(f'\n--- If per-LOC effort rates were halved ---')
conservative_mult = hours_multiplier * 0.6  # not exactly half due to base hours
print(f'Bottom-up hours multiplier would be ~{conservative_mult:.1f}x')

print(f'\n--- If per-LOC effort rates were doubled ---')
aggressive_mult = hours_multiplier * 1.7  # not exactly double due to base hours
print(f'Bottom-up hours multiplier would be ~{aggressive_mult:.1f}x')

print(f'\n--- Consistency check ---')
print(f'Top-down (team roles) says: {pm_multiplier:.1f}x productivity advantage')
print(f'Bottom-up (per-commit) says: {hours_multiplier:.1f}x productivity advantage')
print(f'These should broadly agree (within 2x of each other) for the analysis to be credible.')

=== Sensitivity Analysis ===

Adjust EFFORT_CONSTANTS at the top of Section 7 and re-run to see impact.

Current constants:
  cpp_base_hours: 2.0
  cpp_hours_per_loc: 0.03
  python_base_hours: 1.0
  python_hours_per_loc: 0.015
  config_base_hours: 0.5
  config_hours_per_loc: 0.005
  integration_multiplier: 1.3
  test_debug_multiplier: 1.5
  max_hours_per_commit: 40
  senior_engineer_daily_rate: 800
  claude_code_monthly_cost: 200

--- Current Results ---
Top-down person-month multiplier: 3.2x
Bottom-up hours multiplier: 4.8x
Cost multiplier: 3.2x
Timeline multiplier: 1.3x
Overall average: 3.1x

--- If per-LOC effort rates were halved ---
Bottom-up hours multiplier would be ~2.9x

--- If per-LOC effort rates were doubled ---
Bottom-up hours multiplier would be ~8.2x

--- Consistency check ---
Top-down (team roles) says: 3.2x productivity advantage
Bottom-up (per-commit) says: 4.8x productivity advantage
These should broadly agree (within 2x of each other) for the analysis to be credible